# Compare reconstructions — grid

Evaluates reconstruction consistency across random instantiations of the reconstruction algorithm (e.g. STRND with different random seeds). A grid is drawn on one reference reconstruction and the same barcodes are painted in every other reconstruction. If the reconstructions agree, the grid appears (up to rotation, reflection and scale) in all of them.

Images are saved to `$SPAN_DATA_ROOT/scratch/<project>/eval_grid/`.

## Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import time
from time import perf_counter
import random
import pickle
from collections import deque
from tqdm.auto import tqdm
from graph_tool import Graph
from graph_tool.draw import sfdp_layout, random_layout
from graph_tool.topology import label_components
import graph_tool as gt
import matplotlib.pyplot as plt

%config InlineBackend.figure_format = 'retina'

print(gt.__file__)
print("Version: ", print(gt.__version__))
print("OpenMP Enabled: ", gt.openmp_enabled())

if gt.openmp_enabled():
    print("Number of threads before: ", gt.openmp_get_num_threads())
    gt.openmp_set_num_threads(1)
    print("Number of threads after: ", gt.openmp_get_num_threads())

## Helper functions

In [ ]:
# ----------------------------
# Helpers
# ----------------------------
def normalize_pos_95(pos):
    """
    pos: (N,2) float array
    Normalize so mean -> (0,0) and central 95% per-axis lies within [-1,1]
    using a single isotropic scale factor.
    Returns: (pos_norm, finite_mask)
    """
    pos = np.asarray(pos, dtype=np.float64)
    if pos.ndim != 2 or pos.shape[1] != 2:
        raise ValueError(f"pos must be Nx2; got {pos.shape}")

    finite = np.isfinite(pos).all(axis=1)
    pf = pos[finite]
    if pf.shape[0] == 0:
        raise ValueError("No finite positions to normalize.")

    # center
    mu = pf.mean(axis=0)
    p0 = pf - mu

    # scale (95% per-axis) with isotropic factor
    qlo = np.percentile(p0, 2.5, axis=0)
    qhi = np.percentile(p0, 97.5, axis=0)
    axis_halfspan = np.maximum(np.abs(qlo), np.abs(qhi))
    s = axis_halfspan.max()
    if not np.isfinite(s) or s == 0:
        raise ValueError("Degenerate layout: cannot determine nonzero scale.")

    pos_norm = np.full_like(pos, np.nan)
    pos_norm[finite] = p0 / s

    # --- Quick sanity checks (still NumPy-fast) ---
    mean_xy = pos_norm.mean(axis=0)
    qlo2 = np.percentile(pos_norm, 2.5, axis=0)
    qhi2 = np.percentile(pos_norm, 97.5, axis=0)
    
    print("mean (should be ~0,0):", mean_xy)
    print("2.5% quantiles:", qlo2, "  97.5% quantiles:", qhi2)
    print("max |quantile| per-axis:", np.maximum(np.abs(qlo2), np.abs(qhi2)))
    
    return pos_norm, finite

def barcodes_on_lines_from_pos_norm(
    g,
    pos_norm,                 # Nx2 numpy array
    barcode_prop="barcode",
    line_spacing_nm=200,
    line_width_nm=15,
    unit_um=2.5,              # 1.0 units == 2.5 um
    include_vertical=True,
    include_horizontal=True,
    square_min=-1.0,
    square_max=1.0,
):
    pos_norm = np.asarray(pos_norm, dtype=np.float64)
    if pos_norm.ndim != 2 or pos_norm.shape[1] != 2:
        raise ValueError(f"pos_norm must be Nx2; got {pos_norm.shape}")

    bc = g.vp["barcode"]

    x = pos_norm[:, 0]
    y = pos_norm[:, 1]

    # nm -> units
    nm_per_unit = unit_um * 1000.0
    spacing = line_spacing_nm / nm_per_unit
    width   = line_width_nm   / nm_per_unit
    half_w  = 0.5 * width

    if spacing <= 0 or width <= 0:
        raise ValueError("line_spacing_nm and line_width_nm must be positive.")

    # line centers include 0 and cover [-1,1] (plus half-width margin)
    k_min = int(np.floor((square_min - half_w) / spacing))
    k_max = int(np.ceil ((square_max + half_w) / spacing))
    centers = np.arange(k_min, k_max + 1, dtype=np.float64) * spacing

    # membership mask: within half-width of any stripe center
    on_any = np.zeros(x.shape[0], dtype=bool)

    if include_vertical:
        dx_min = np.min(np.abs(x[:, None] - centers[None, :]), axis=1)
        on_any |= (dx_min <= half_w)

    if include_horizontal:
        dy_min = np.min(np.abs(y[:, None] - centers[None, :]), axis=1)
        on_any |= (dy_min <= half_w)

    # keep only points inside the square
    inside = (x >= square_min) & (x <= square_max) & (y >= square_min) & (y <= square_max)
    on_any &= inside

    idx = np.flatnonzero(on_any)

    # barcodes are strings: extract only selected vertices (small Python loop)
    barcodes = [bc[g.vertex(int(i))] for i in idx]

    return idx, barcodes, on_any, centers, spacing, width

def left_vs_right_from_barcode(g, barcode_prop="barcode"):
    bc = g.vp[barcode_prop]
    lr = np.fromiter(
        (
            1 if (
                (x.decode("utf-8", "replace") if isinstance(x, (bytes, bytearray)) else str(x))
                .strip().removeprefix("b'").removesuffix("'")
                .startswith("L")
            ) else 0
            for x in (bc[v] for v in g.vertices())
        ),
        dtype=np.uint8,
        count=g.num_vertices(),
    )
    print("L count:", int(lr.sum()), "R/other count:", int(lr.size - lr.sum()))
    return lr

def plot_graph_with_black_hits(
    g,
    pos_norm,                 # Nx2 normalized positions
    left_or_right,            # length N array, 0/1
    bcs,                      # list of barcode strings from earlier
    out_png,
    dpi=1000,
    marker_size=0.25,
    title=None,
):
    pos_norm = np.asarray(pos_norm, dtype=np.float64)
    if pos_norm.ndim != 2 or pos_norm.shape[1] != 2:
        raise ValueError(f"pos_norm must be Nx2; got {pos_norm.shape}")

    N = pos_norm.shape[0]
    left_or_right = np.asarray(left_or_right, dtype=np.uint8)
    if left_or_right.shape[0] != N:
        raise ValueError(f"left_or_right length {left_or_right.shape[0]} != N {N}")

    finite = np.isfinite(pos_norm).all(axis=1)
    mask0 = (left_or_right == 0)
    mask1 = (left_or_right == 1)

    # Determine which vertices are "hits" by barcode membership in bcs
    # (barcode strings => direct lookup; O(N) property reads)
    bcs_set = set(bcs)
    bc = g.vp.barcode

    hit = np.zeros(N, dtype=bool)
    for i in range(N):
        if not finite[i]:
            continue
        hit[i] = (bc[g.vertex(i)] in bcs_set)

    nonhit0 = mask0 & finite & ~hit
    nonhit1 = mask1 & finite & ~hit
    hit &= finite

    plt.figure(figsize=(6, 6))
    if title:
        plt.title(title)

    # non-hits
    plt.scatter(pos_norm[nonhit0, 0], pos_norm[nonhit0, 1],
                s=marker_size, marker='o', facecolors='none', edgecolors='blue', linewidths=marker_size/2)
    plt.scatter(pos_norm[nonhit1, 0], pos_norm[nonhit1, 1],
                s=marker_size, marker='o', facecolors='none', edgecolors='red', linewidths=marker_size/2)

    # hits on top
    plt.scatter(pos_norm[hit, 0], pos_norm[hit, 1],
                s=marker_size, marker='o', facecolors='black', edgecolors='black', linewidths=marker_size/2)

    plt.xlim(-1.2, 1.2); plt.ylim(-1.2, 1.2)
    plt.gca().set_aspect("equal", adjustable="box")
    plt.xticks([]); plt.yticks([]); plt.box(False)

    plt.savefig(out_png, dpi=dpi, bbox_inches="tight")
    plt.close()
    print("Saved:", out_png, "| total:", N, "finite:", finite.sum(),
          "mask0:", mask0.sum(), "mask1:", mask1.sum(), "hits:", hit.sum())

def load_pos_from_csv(csv_path):
    df_pos = pd.read_csv(csv_path)

    # --- build pos2 from dataframe, aligned to original graph node IDs ---
    # Expect columns: node_ID, x, y
    required = {"node_ID", "x", "y"}
    missing = required - set(df_pos.columns)
    if missing:
        raise ValueError(f"CSV missing columns {sorted(missing)}. Found: {list(df_pos.columns)}")
    
    # Sort by node_ID and construct (N,2) array
    df_pos = df_pos.sort_values("node_ID", kind="mergesort").reset_index(drop=True)
    
    node_ids = df_pos["node_ID"].to_numpy()
    if node_ids.size == 0:
        raise ValueError("Positions CSV is empty.")
    
    # Basic checks: integer-ish, unique, contiguous (optional but recommended)
    if not np.issubdtype(node_ids.dtype, np.integer):
        # try to coerce
        node_ids = pd.to_numeric(df_pos["node_ID"], errors="raise").astype(np.int64).to_numpy()
    
    if len(np.unique(node_ids)) != len(node_ids):
        raise ValueError("node_ID column contains duplicates.")
    
    # If you want strict alignment with graph-tool internal IDs:
    # this assumes node_ID covers 0..N-1 with no gaps
    expected = np.arange(node_ids.min(), node_ids.min() + len(node_ids), dtype=node_ids.dtype)
    if not np.array_equal(node_ids, expected):
        raise ValueError(
            "node_IDs are not contiguous after sorting. "
            f"min={node_ids.min()}, max={node_ids.max()}, count={len(node_ids)}"
        )

    pos = df_pos[["x", "y"]].to_numpy(dtype=np.float64, copy=False)

    return pos

## Parameters and graph
The graph is only used for barcode identities and the L/R class of each vertex.

In [ ]:
# Parameters
project_name = "260921Jac"
sample_name = "C1"

# Suffix of the graph that was reconstructed in pipeline step 07
suffix = "postPipeline0302_Lev3_LCC2_min_e2_min_d2_max_d30_LCC"

data_dir = os.path.join(os.environ.get("SPAN_DATA_ROOT", "./orcd"), "scratch", project_name)

filename = f"{data_dir}/{sample_name}_barcodesGraph_{suffix}.gt.gz"

print("Loading: ", filename)

g = gt.load_graph(filename)

In [ ]:
num_nodes = g.num_vertices()
num_edges = g.num_edges()

print(f"Number of nodes: {num_nodes:,}")
print(f"Number of edges: {num_edges:,}")

In [ ]:
left_or_right = left_vs_right_from_barcode(g)

#left_or_right = np.random.choice([0, 1], size=g.num_vertices(), p=[0.333, 0.667])

## Reconstructions to compare

In [ ]:
# Reconstructions to compare, by the tag in their step-07 file names
# (<SAMPLE>_positions_<SUFFIX>_STRND_<tag>.csv, tag = d<node_embedding_components>_rnd<random_state>).
# The first one is the reference.
strnd_layouts_to_compare = [
    "d64_rnd42",
    "d64_rnd999", 
    "d64_rnd123"
]

## Paint a grid from the reference onto the others
In the reference reconstruction, vertices lying on a grid of lines (`line_spacing_nm`, `line_width_nm`) are selected and drawn black; the same barcodes are then drawn black in every other reconstruction. Remaining vertices are blue (R) or red (L). `unit_um` in `barcodes_on_lines_from_pos_norm` sets the physical size of one normalized unit.

In [ ]:
dpi=500
os.makedirs(f"{data_dir}/eval_grid", exist_ok=True)
marker_size=0.1

for i in range(len(strnd_layouts_to_compare) - 1):
    ref_layout = strnd_layouts_to_compare[i]

    print()
    print(f"Using STRND_{ref_layout} as reference")

    csv_path = f"{data_dir}/{sample_name}_positions_{suffix}_STRND_{ref_layout}.csv"

    print("Loading: ", csv_path)
    
    pos = load_pos_from_csv(csv_path)

    # Normalize (mean -> 0, 95% -> [-1,1])
    pos_norm, finite = normalize_pos_95(pos)
    print(pos_norm.shape)

    idx, bcs, mask, centers, spacing_u, width_u = barcodes_on_lines_from_pos_norm(
        g, pos_norm,
        line_spacing_nm=1500,
        line_width_nm=250,
        include_vertical=True,
        include_horizontal=True,  
    )
    
    print("spacing (units):", spacing_u, " width (units):", width_u)
    print("hits:", len(bcs))
    
    # Plot
    out_filename = f"{data_dir}/eval_grid/{sample_name}_{suffix}_ref{ref_layout}.png"
    plot_graph_with_black_hits(
        g,
        pos_norm,
        left_or_right, 
        bcs=bcs,       
        out_png=out_filename,
        dpi=dpi,
        marker_size=marker_size,
        title=None
    )

    for j in range(i + 1, len(strnd_layouts_to_compare)):
        comp_layout = strnd_layouts_to_compare[j]

        print()
        print(f"Comparing to STRND_{comp_layout}")
    
        csv_path = f"{data_dir}/{sample_name}_positions_{suffix}_STRND_{comp_layout}.csv"
    
        print("Loading: ", csv_path)

        pos2 = load_pos_from_csv(csv_path)

        # Normalize (mean -> 0, 95% -> [-1,1])
        pos2_norm, finite2 = normalize_pos_95(pos2)
        
        # Plot
        out_filename = f"{data_dir}/eval_grid/{sample_name}_{suffix}_ref{ref_layout}_comp{comp_layout}.png"
        plot_graph_with_black_hits(
            g,
            pos2_norm,
            left_or_right, 
            bcs=bcs,       
            out_png=out_filename,
            dpi=dpi,
            marker_size=marker_size,
            title=None
        )